In [1]:
%pwd

'/home/emanu/Documents/PycharmProjects/HPEClinicalEvaluation'

In [2]:
import h5py

In [3]:
# Load arrays from the HDF5 file
gt_keypoints_all = []
pred_keypoints_all = []
inference_times_all = []

with h5py.File('data_arrays.h5', 'r') as hf:
    for i in range(4):
        gt_keypoints_all.append(hf[f'gt_keypoints_all_{i}'][:])
        pred_keypoints_all.append(hf[f'pred_keypoints_all_{i}'][:])
        inference_times_all.append(hf[f'inference_times_all_{i}'][:])

print("Loaded gt_keypoints_all shapes:", [arr.shape for arr in gt_keypoints_all])
print("Loaded pred_keypoints_all shapes:", [arr.shape for arr in pred_keypoints_all])
print("Loaded inference_times_all shapes:", [arr.shape for arr in inference_times_all])

Loaded gt_keypoints_all shapes: [(2201, 16, 3), (1700, 16, 3), (2026, 16, 3), (1703, 16, 3)]
Loaded pred_keypoints_all shapes: [(2201, 16, 3), (1700, 16, 3), (2026, 16, 3), (1703, 16, 3)]
Loaded inference_times_all shapes: [(2201,), (1700,), (2026,), (1703,)]


In [4]:
import numpy as np
import plotly.graph_objects as go
from utils import metrics
from utils.plotKeypoints import plot_3d_keypoints_gt_pred

In [5]:
inference_times_all = np.concatenate(inference_times_all)
gt_keypoints_all = np.concatenate(gt_keypoints_all, axis=0)
pred_keypoints_all = np.concatenate(pred_keypoints_all, axis=0)

In [8]:
target = gt_keypoints_all
prediction = pred_keypoints_all

In [9]:
assert prediction.shape == target.shape

In [10]:
mean_inf = np.mean(inference_times_all)
std_inf = np.std(inference_times_all)
print('Mean Inf: ' + str(1/mean_inf) + ', Std Inf: ' + str(1/std_inf))

Mean Inf: 7.873312277763365, Std Inf: 119.60183695640634


In [34]:
frame_idx = 5001
target_single = target[frame_idx]
prediction_single = prediction[frame_idx] 

colors = ['green', 'red']
names = ['Ground Truth KeyPoints', 'HPE Pred KeyPoints']
model_name = 'mediapipe'
# Define connections between related keypoints
if model_name == 'mediapipe':
    connections = [(0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (6, 7), (0, 6),
                    (1, 7), (6, 8), (7, 9), (8, 10), (9, 11), (10, 12),
                    (11, 13), (10, 14), (11, 15), (12, 14), (13, 15)]

# Create a Plotly 3D scatter plot
fig = go.Figure()

all_values = ()
idx = 0
for keypoints in [target_single, prediction_single]:
    # Extract X, Y, and Z coordinates from keypoints
    x, y, z = zip(*keypoints)

    # Scatter plot for keypoints
    #fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=colors[idx], size=5)))
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(color=colors[idx], size=5), 
                                text=[f"{idx} {i}" for i in range(len(x))], hoverinfo='text'))

    # Plot connections
    
    # Plot connections
    for connection in connections:
        x_vals = [x[connection[0]], x[connection[1]]]
        y_vals = [y[connection[0]], y[connection[1]]]
        z_vals = [z[connection[0]], z[connection[1]]]
        fig.add_trace(go.Scatter3d(x=x_vals, y=y_vals, z=z_vals, mode='lines', line=dict(color=colors[idx])))

    idx += 1

    # Combine x, y, z values into a single list
    all_values = all_values + x + y + z

# Find the minimum and maximum values
min_value = min(all_values)
max_value = max(all_values)

# Update layout to set axis limits
fig.update_layout(
    scene=dict(
        aspectmode='cube',
        xaxis=dict(title='X', range=[-100, max_value]),
        yaxis=dict(title='Y', range=[-100, max_value]),
        zaxis=dict(title='Z', range=[min_value, max_value])
    )
)

fig.show()

In [ ]:
    mpjpe = np.sqrt(np.sum((prediction - target) ** 2, axis=1))
    mean = np.mean(mpjpe)
    std = np.std(mpjpe)
    print('Mean Inf: ' + str(mean) + ', Std Inf: ' + str(std))

In [ ]:
transposed = False
if prediction.shape[0] != 3 and prediction.shape[0] != 2:
    prediction = prediction.T
    target = target.T
    transposed = True
assert (target.shape[1] == prediction.shape[1])

muX = np.mean(target, axis=1, keepdims=True)
muY = np.mean(prediction, axis=1, keepdims=True)

X0 = target - muX
Y0 = prediction - muY

normX = np.sqrt(np.sum(X0 ** 2, axis=(1, 2), keepdims=True))
normY = np.sqrt(np.sum(Y0 ** 2, axis=(1, 2), keepdims=True))

X0 /= normX
Y0 /= normY

H = np.matmul(X0.transpose(0, 2, 1), Y0)

try:
    U, _, V = np.linalg.svd(H)
except:
    mean = np.nan
    std = np.nan

u, s, v_t = np.linalg.svd(H)
v = v_t.transpose(0, 2, 1)
r = np.matmul(v, u.transpose(0, 2, 1))

# Avoid improper rotations (reflections), i.e. rotations with det(R) = -1
sign_det_r = np.sign(np.expand_dims(np.linalg.det(r), axis=1))
v[:, :, -1] *= sign_det_r
s[:, -1] *= sign_det_r.flatten()
r = np.matmul(v, u.transpose(0, 2, 1))  # Rotation

tr = np.expand_dims(np.sum(s, axis=1, keepdims=True), axis=2)

a = tr * normX / normY  # Scale
t = muX - a * np.matmul(muY, r)  # Translation

# Perform rigid transformation on the input
predicted_aligned = a * np.matmul(prediction, r) + t

# Return MPJPE
mean = np.mean(np.linalg.norm(predicted_aligned - target, axis=len(target.shape) - 1))
std = np.std(np.linalg.norm(predicted_aligned - target, axis=len(target.shape) - 1))
